<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/main/AgenticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Math category

- chiama i tool se c'è un'espressione già fatta. non rielabora le informazioni per creare l'espressione
- il cross encoder non prende la risposta bene in questo caso
- tool da aggiungere:
  - qualcosa per gestire le percentuali
  - ricerca web

In [1]:
from google.colab import userdata
from huggingface_hub import login
import os
import sys
import time

In [2]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [3]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Repository clone...
Cloning into 'NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi'...
remote: Enumerating objects: 272, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 272 (delta 59), reused 14 (delta 4), pack-reused 168 (from 1)
Receiving objects: 100% (272/272), 18.87 MiB | 7.46 MiB/s, done.
Resolving deltas: 100% (138/138), done.
/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi


In [4]:
API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Logged in as: GliEmbeddingRuspanti (role: student)


# Model Qwen

In [5]:
!pip install -q transformers accelerate bitsandbytes langchain langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.6 MB/s eta 0:00:00


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer


In [7]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [8]:
final_model_name = "Qwen/Qwen2.5-7B-Instruct"
final_model = AutoModelForCausalLM.from_pretrained(
    final_model_name,
    device_map="auto",
    torch_dtype="auto"
)
final_tokenizer = AutoTokenizer.from_pretrained(final_model_name)
final_tokenizer.pad_token = final_tokenizer.eos_token

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

# Agentic

##Tools

In [9]:
import math
from langchain.tools import tool
from sympy import sympify, Eq, solve
import re


In [10]:
@tool
def empty_tool() -> str:
  """
  An empty tool that does nothing. Use in case the other tools are not useful
  """
  return ""

In [11]:
@tool("calculator")
def calculator(expression: str) -> str:
    """
    Performs arithmetic calculations. Evaluate mathematical expressions.
    """
    return str(eval(expression))

In [12]:
@tool
def solve_equation(expressions)-> str:
      """
      Solve a mathematical equation or a system of mathematical equations.
      """

      expressions = [expressions]

      equations = []
      symbols_set = set()

      variable_names = set()

      for expr in expressions:
          variable_names.update(
              re.findall(r"[a-zA-Z_]\w*", expr)
          )

      symbols_dict = {
          name: symbols(name)
          for name in variable_names
      }

      for expr in expressions:

          if "=" in expr:

              left, right = expr.split("=")

              eq = Eq(
                  eval(left, {}, symbols_dict),
                  eval(right, {}, symbols_dict)
              )

          else:

              eq = Eq(
                  eval(expr, {}, symbols_dict),
                  0
              )

          equations.append(eq)

          symbols_set.update(eq.free_symbols)

      variables = list(symbols_set)

      result = solve(equations, variables)

      return str(result)

In [48]:
tools = [calculator, solve_equation]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
calculator
Performs arithmetic calculations. Evaluate mathematical expressions.
{'expression': {'title': 'Expression', 'type': 'string'}}
--
solve_equation
Solve a mathematical equation or a system of mathematical equations.
{'expressions': {'title': 'Expressions'}}


In [14]:
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

empty_tool() -> str - An empty tool that does nothing. Use in case the other tools are not useful
calculator(expression: str) -> str - Performs arithmetic calculations. Evaluate mathematical expressions.
solve_equation(expressions) -> str - Solve a mathematical equation or a system of mathematical equations.


## llm creation

In [15]:
!pip install langchain_huggingface

In [16]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer,pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

In [17]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [18]:
llm = HuggingFacePipeline(
    pipeline=pipe,
    pipeline_kwargs={
        "max_new_tokens": 150,
        "temperature": 0
    }
)

In [19]:
final_pipe = pipeline(
    "text-generation",
    model=final_model,
    tokenizer=final_tokenizer
)

In [20]:
final_llm = HuggingFacePipeline(
    pipeline=final_pipe,
    pipeline_kwargs={
        "max_new_tokens": 30,
        "temperature": 0
    }
)

## Example to see if it works

In [15]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant."
    ),
    (
        "user",
        "{input}"
    )
])

In [16]:
chain = prompt | llm

In [17]:
response = chain.invoke({
    "input": "What are the capitals of Australia, Canada, Brazil and South Africa?"
})
print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


System: You are a helpful assistant.
Human: What are the capitals of Australia, Canada, Brazil and South Africa? Please provide the names in English.

Assistant: The capitals of Australia, Canada, Brazil and South Africa are:
Australia - Canberra
Canada - Ottawa
Brazil - Brasília
South Africa - Pretoria

Note that these are not necessarily the largest cities within each country. For example, Sydney is the most populous city in Australia, while São Paulo is the most populous city in Brazil. Similarly, Toronto is the most populous city in Canada, while Cape Town is the capital but not the most populous city in South Africa. Also, Brasília is the capital of Brazil, but it was established as a new capital to replace Rio de Janeiro after the 1960s military coup d'état which led to the abolition of the federal republic constitution, making Rio de Janeiro an exclusive status as the seat of government for the Federation of Brazil. The current Constitution re-established Rio de Janeiro's role a

### Prompts

In [24]:
system_prompt_router = f"""\
You are a correct assistant that has access only to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return only the name and input of the tool to use. Choose the best tool among the ones above only. It has to be useful for the user request
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding to the argument names and the values corresponding to the values requested by the user.
Give this JSON after Assistant:
If there is no useful tool among the ones above, redirect to the empty_tool
"""

In [25]:
from langchain_core.runnables import RunnablePassthrough

prompt_router = ChatPromptTemplate.from_messages(
    [("system", system_prompt_router), ("user", "{input}")]
)


In [26]:
final_prompt = ChatPromptTemplate.from_template("""
You are an helpful and correct assistant
Question:
{input}

Tool used:
{tool_call}

Tool output:
{output}

Answer the user naturally. Use the tool output to generate the answer, but in the answer just answer the question without the thought procedure.
""")

### Helpers for chain

In [27]:
import json
def extract_json_after_assistant(text: str):

    decoder = json.JSONDecoder()

    candidate = text.split("Assistant:")[-1]

    start = candidate.find("{")

    obj, _ = decoder.raw_decode(candidate[start:])

    return obj

In [28]:
from langchain_core.runnables import RunnableLambda

custom_parser = RunnableLambda(extract_json_after_assistant)

In [29]:
from typing import Any, Dict, Optional, TypedDict
from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""
    name: str
    arguments: Dict[str, Any]


def invoke_tool( x: Dict[str, Any], config: Optional[RunnableConfig] = None ):
    """
    A function that we can use the perform a tool invocation safely.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool

    Falls back to empty_tool if:
        tool_call is missing
        name is missing
        tool does not exist
        arguments are malformed
    """

    tool_name_to_tool = {
        tool.name: tool
        for tool in tools
    }

    try:

        # recupera tool_call
        tool_call_request = x.get("tool_call", {})

        # recupera nome tool
        name = tool_call_request.get(
            "name",
            "empty_tool"
        )

        # se il tool non esiste -> empty_tool
        if name not in tool_name_to_tool:
            return empty_tool.invoke({}, config=config)

        requested_tool = tool_name_to_tool[name]

        # recupera arguments
        arguments = tool_call_request.get(
            "arguments",
            {}
        )

        # se arguments non è un dict -> empty_tool
        if not isinstance(arguments, dict):
            return empty_tool.invoke({}, config=config)

        # esegui tool richiesto
        return requested_tool.invoke(
            arguments,
            config=config
        )

    except Exception:

        # fallback totale
        return empty_tool.invoke({}, config=config)

### Chain

In [30]:
router_chain = (
    {
        "input": RunnableLambda(lambda x: x["input"]),

        "tool_call":
            prompt_router
            | llm
            | custom_parser
    }
)

In [31]:
chain = (
    router_chain
    | RunnablePassthrough.assign(
        output=invoke_tool
    )
    | final_prompt
    | llm
)

In [32]:
response = chain.invoke({"input": "What is the result of 2+2?"})
print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: 
You are an helpful and correct assistant
Question:
What is the result of 2+2?

Tool used:
{'name': 'calculator', 'arguments': {'expression': '2 + 2'}}

Tool output:
4

Answer the user naturally. Use the tool output to generate the answer, but in the answer just answer the question without the thought procedure.
The result of 2+2 is 4.

Assistant: The result of 2+2 is 4. This can be verified using a calculator or simple addition.


# Model

In [49]:
from sentence_transformers import CrossEncoder

In [50]:
modelCrossencoder = CrossEncoder('cross-encoder/stsb-distilroberta-base', trust_remote_code=True)

'''
def pick_by_crossencoder(model_output: str, options: dict):
    labels = list(options.keys())
    roberta_inputs = [[model_output, options[l]] for l in labels]
    scores = modelCrossencoder.predict(roberta_inputs)
    best_label = labels[int(np.argmax(scores))]
    return best_label, {labels[i]: float(scores[i]) for i in range(len(labels))}
'''

def pick_by_crossencoder(model_output: str, options: dict):
    labels = list(options.keys())
    pairs = [[model_output, options[l]] for l in labels]

    scores = modelCrossencoder.predict(pairs)

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = float(best_score - np.mean(sorted_scores[1:])) if len(labels) > 1 else 0.0

    # --- NORMALIZED MARGIN (probabilistic closeness of 2nd to 1st) ---
    score_range = np.max(scores) - np.min(scores) + 1e-8
    normalized_margin = (best_score - second_score) / score_range

    # interpretazione probabilistica del gap (sigmoid-like)
    relative_second_closeness = np.exp(second_score) / (np.exp(best_score) + np.exp(second_score))

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i]) for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i]) for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
import numpy as np
from typing import Callable

In [52]:
class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, options, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"

In [57]:
from typing import Callable, Any, Dict, Optional, TypedDict
import json

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableConfig
)
from langchain_core.tools import render_text_description


class AgenticModel(Model):

    def __init__(
        self,
        name: str,
        router_llm,
        final_llm,
        tools,
        answer_fn: Callable
    ):

        super().__init__(name, answer_fn)

        self.router_llm = router_llm
        self.final_llm = final_llm
        self.tools = tools

        # =========================
        # TOOL DESCRIPTIONS
        # =========================

        self.rendered_tools = render_text_description(
            self.tools
        )

        # =========================
        # ROUTER SYSTEM PROMPT
        # =========================
        self.system_prompt_router = f"""\
            You are a correct assistant that has access ONLY to the following tools.
            Here are the names and descriptions for each tool:

            {self.rendered_tools}

            Your task is to select the BEST tool.
            Given the user input, return only the name and input of the tool to use.
            Choose the best tool among the ones above only.
            It has to be useful for the user request.
            Return your response as a JSON blob with 'name' and 'arguments' keys.

            The `arguments` should be a dictionary, with keys corresponding to the argument names and the values corresponding to the values requested by the user.
            Give this JSON after `Assistant:`
            If there is no useful tool among the ones above, redirect to the tool with name empty_tool and arguments empty dictionary.
            """

        # =========================
        # ROUTER PROMPT
        # =========================

        self.prompt_router = ChatPromptTemplate.from_messages(
            [
                ("system", self.system_prompt_router),
                ("user", "{input} and the possible answers are {options}")
            ]
        )

        # =========================
        # FINAL PROMPT
        # =========================
        self.final_prompt = ChatPromptTemplate.from_template("""
        You are an helpful and correct assistant answering to a MCA question.
        Question:
        {input}

        Possible answers:
        {options}

        Tool used:
        {tool_call}

        Tool output:
        {output}

        Answer the user naturally. Use the tool output to generate the answer.
        Do NOT explain your reasoning process.
        Answer with the option text, not the number
        """)

        # =========================
        # PARSER
        # =========================

        self.custom_parser = RunnableLambda(
            self.extract_json_after_assistant
        )

        # =========================
        # ROUTER CHAIN
        # =========================
        self.router_chain = (
            {
                "input": RunnableLambda(
                    lambda x: x["input"]
                ),
                "options": RunnableLambda(
                    lambda x: x["options"]
                ),
                "tool_call":
                    self.prompt_router
                    | self.router_llm
                    | self.custom_parser
            }
        )

        # =========================
        # FULL CHAIN
        # =========================

        self.chain = (
            self.router_chain
            | RunnablePassthrough.assign(
                output=self.invoke_tool
            )
            | self.final_prompt
            | self.final_llm
        )

    # ==================================
    # JSON PARSER
    # ==================================
    def extract_json_after_assistant(self, text: str):
        try:

            decoder = json.JSONDecoder()

            candidate = text.split("Assistant:")[-1]

            start = candidate.find("{")

            obj, _ = decoder.raw_decode(candidate[start:])

            return obj

        except Exception:
            return {
                "name": "empty_tool",
                "arguments": {}
            }

    # ==================================
    # TOOL INVOCATION
    # ==================================
    def invoke_tool(
        self,
        x: Dict[str, Any],
        config: Optional[RunnableConfig] = None
    ):
        """
        A function that we can use the perform a tool invocation safely.

        Args:
            tool_call_request: a dict that contains the keys name and arguments.
                The name must match the name of a tool that exists.
                The arguments are the arguments to that tool.
            config: This is configuration information that LangChain uses that contains
                things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

        Returns:
            output from the requested tool

        Falls back to empty_tool if:
            tool_call is missing
            name is missing
            tool does not exist
            arguments are malformed
        """
        tool_name_to_tool = {
            tool.name: tool
            for tool in self.tools
        }

        try:

            tool_call_request = x.get(
                "tool_call",
                {}
            )

            name = tool_call_request.get(
                "name",
                "empty_tool"
            )

            if name not in tool_name_to_tool:

                return empty_tool.invoke({})

            requested_tool = tool_name_to_tool[
                name
            ]

            arguments = tool_call_request.get(
                "arguments",
                {}
            )

            if not isinstance(arguments, dict):

                return empty_tool.invoke({})

            return requested_tool.invoke(
                arguments,
                config=config
            )

        except Exception:

            return empty_tool.invoke({})

    # ==================================
    # GENERATE
    # ==================================

    def generate(
        self,
        question: str,
        options: dict,
        system_prompt: str = ""
    ) -> str:

        response = self.chain.invoke(
            {
                "input": question,
                "options": options
            }
        )

        # HuggingFacePipeline può restituire stringa
        # oppure AIMessage

        if hasattr(response, "content"):

            return response.content

        return str(response)

# Game

In [58]:
def play_game(game, model_name, verbose=False):
    #models = models_configs["models"]
    #system_prompt = models_configs["system_prompt"]
    agenticModel = AgenticModel(model_name, llm, llm, tools, pick_by_crossencoder)
    log = []

    while game.in_progress:
        question = game.current_question
        if not question:
            print("No question available. Game may have ended.")
            break

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question.text}")
        for opt in question.options:
            print(f"  [{opt.id}] {opt.text}")

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        options = {f"{opt.id}": opt.text for opt in question.options}

        t0 = time.time()
        answer_summary, answer_input = agenticModel.answer(question.text, options)
        #answer_summary, answer_input = answer_ensemble(
         #   models, question.text, options, system_prompt, verbose=verbose
        #)
        inference_time = time.time() - t0

        print(f"Agentic answer: {answer_input}")
        answer_id = int(answer_input)
        choosen_answer = question.options[answer_id]

        result = game.answer(answer_id)

        if result.correct:
            print(" CORRECT!")
            if result.game_over:
                print(f"\n CONGRATULATIONS! You completed the game!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")
        elif result.timed_out:
            print("TIMED OUT!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        elif not result.correct:
            print(" WRONG ANSWER!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        # save outcome in the log(useful for graphs)
        entry = {
            'level'          : game.current_level,
            'question'       : question.text,
            'options'        : question.options,
            'chosen_option'  : choosen_answer.text,
            'correct'        : result.correct,
            'timed_out'      : result.timed_out,
            'inference_time' : round(inference_time, 2),
            #'answer_summary' : answer_summary,
        }
        log.append(entry)

    #ensemble_name = " + ".join(m.name for m in models)

    summary = {
        'model'          : model_name,
        'final_level'    : game.current_level,
        'earned_amount'  : game.earned_amount,
        'num_questions'  : len(log),
        'num_correct'    : sum(1 for e in log if e['correct']),
        'num_timed_out'  : sum(1 for e in log if e['timed_out']),
        'avg_inference_s': round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'            : log,
    }

    print(f"\n=== Game Summary ===")
    #print(f"Ensemble  : {ensemble_name}")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

    return summary

In [60]:
model_name="agenticModel"
print(f"\n########## MODEL: {model_name} ##########")

    #model = config["model"]
model_results = []
for comp_id in [3]:
    print(f"\n--- Competition {comp_id} ---")

    game = client.game.start(competition_id=comp_id)

    summary = play_game(game, model_name)

    model_results.append(summary)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



########## MODEL: agenticModel ##########

--- Competition 3 ---

--- Level 1 ---
Q: A filling machine puts an average of four ounces of coffee in jars, with a standard deviation of 0.25 ounces. Forty jars filled by this machine are selected at random. What is the probability that the mean amount per jar filled in the sampled jars is less than 3.9 ounces?
  [0] 0.025
  [1] 0.0225
  [2] 0.0057
  [3] 0.05

Time remaining: 30.0s


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL ANSWER ----->Human: 
        You are an helpful and correct assistant answering to a MCA question.
        Question:
        A filling machine puts an average of four ounces of coffee in jars, with a standard deviation of 0.25 ounces. Forty jars filled by this machine are selected at random. What is the probability that the mean amount per jar filled in the sampled jars is less than 3.9 ounces?

        Possible answers:
        {'0': '0.025', '1': '0.0225', '2': '0.0057', '3': '0.05'}

        Tool used:
        {}

        Tool output:
        

        Answer the user naturally. Use the tool output to generate the answer.
        Do NOT explain your reasoning process.
        Answer with the option text, not the number
         2

Assistant: 0.0057
Agentic answer: 1
 WRONG ANSWER!

 Game Over! | Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00


In [ ]:
def print_results(results):
        print("\n" + "=" * 80)
        print(f"MODELLO: {model_name}")
        print("=" * 80)

        for i, summary in enumerate(competitions):

            print(f"\n🏁 Competition {i}")
            print("-" * 60)

            print(f"Model name        : {summary['model']}")
            print(f"Final level       : {summary['final_level']}")
            print(f"Earned amount     : €{summary['earned_amount']}")
            print(f"Questions         : {summary['num_questions']}")
            print(f"Correct answers   : {summary['num_correct']}")
            print(f"Timed out         : {summary['num_timed_out']}")
            print(f"Avg inference     : {summary['avg_inference_s']} s")

            accuracy = (
                summary['num_correct'] / summary['num_questions'] * 100
                if summary['num_questions'] > 0 else 0
            )

            print(f"Accuracy          : {accuracy:.1f}%")

            print("\n📋 Question Log")
            print("-" * 60)

            confidence_array = []

            for q_idx, entry in enumerate(summary['log'], start=1):

                status = "✅" if entry['correct'] else "❌"

                if entry.get('timed_out'):
                    status = "⏰"

                # ─────────────────────────────────────────────
                # CONFIDENCE EXTRACTION (robust fallback chain)
                # ─────────────────────────────────────────────
                answer_summary = entry.get("answer_summary", {})

                if isinstance(answer_summary, dict):
                    if "normalized_margin" in answer_summary:
                        conf = answer_summary["normalized_margin"]

                    elif "confidence" in answer_summary:
                        conf = answer_summary["confidence"]

                    else:
                        conf = None
                else:
                    conf = None

                confidence_array.append(conf)

                print(
                    f"{q_idx:02d}. "
                    f"{status} "
                    f"Time: {entry['inference_time']:.2f}s "
                    f"Conf: {conf if conf is not None else 'N/A'}"
                )

            # ─────────────────────────────────────────────
            # PRINT SUMMARY CONFIDENCE ARRAY
            # ─────────────────────────────────────────────
            print("\n📊 Confidence Array:")
            if any(c is not None for c in confidence_array):
                print(confidence_array)
            else:
                print("Confidence not available")

        print("\n")

# final print
print_results(results)